# Part 5 — 원본 vs 정제 데이터 비교 파인튜닝
### 데이터 품질이 모델 성능을 결정한다

> **환경**: RTX 3080 + Part 4 완료 상태  
> **소요 시간**: 정제 5분 + 학습 5분 + 평가 3분 = **약 13분**

---

## 🎯 5단계

| # | 내용 | 시간 |
|---|------|------|
| 1 | GPT-4o-mini로 200건 정제 | ~5분 |
| 2 | 정제 데이터로 재학습 (50스텝) | ~5분 |
| 3 | 3가지 응답 나란히 비교 | ~2분 |
| 4 | LLM Judge 채점 | ~1분 |
| 5 | 최종 비교표 + 저장 | |

---

## ⚠️ 사전 조건
- Part 4 완료: `./exaone4-law-raw/` 어댑터 존재
- Part 3 변수 유지: `model`, `tokenizer`, `CONFIG`, `TEST_QUESTIONS`, `generate_fast`, `baseline_results`, `finetuned_results`
- `.env`에 `OPENAI_API_KEY` 설정

---
## 1️⃣ GPT-4o-mini로 데이터 정제

**정제 규칙**
- 구체적 인명·지명·날짜·금액 제거
- 법률 원칙을 묻는 일반적 질문으로
- 답변에 법조항 명시 + 300~500자

In [ ]:
from openai import OpenAI
from datasets import load_dataset
import json, random

client = OpenAI()  # .env의 OPENAI_API_KEY 자동 사용

# 원본 데이터 로드
ds_orig = load_dataset("jihye-moon/LawQA-Ko", split="train")
q_col, a_col = ds_orig.column_names[:2]

random.seed(42)
indices = random.sample(range(len(ds_orig)), 200)

print(f"✅ 정제할 샘플: {len(indices)}건")

In [ ]:
def refine(question, answer):
    prompt = f"""당신은 대한민국 법률 교육 전문가입니다.
아래 법률 상담 QA를 **법률 원칙 학습용 데이터**로 정제해주세요.

[원본 질문]
{question[:600]}

[원본 답변]
{answer[:1000]}

[정제 규칙]
1. 질문에서 구체적 인명, 지명, 날짜, 금액을 제거하고 법률 원칙을 묻는 일반적 질문으로.
2. 답변에서 특정 사건 사실관계를 제거하고 법조항·법률 원칙 중심으로 재작성.
3. 답변은 300~500자. 반드시 한국어.

JSON 형식으로만 출력: {{"question": "...", "answer": "..."}}"""

    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=800, temperature=0.3,
        )
        text = resp.choices[0].message.content.strip()
        if "```" in text:
            text = text.split("```")[1].replace("json", "").strip()
        return json.loads(text)
    except:
        return None

# 1건 테스트
s = ds_orig[indices[0]]
test_result = refine(s[q_col], s[a_col])
print("📋 정제 테스트:")
print(f"  원본 Q: {s[q_col][:100]}...")
print(f"  정제 Q: {test_result['question'] if test_result else '실패'}")

In [ ]:
import time

# 200건 일괄 정제 (약 5분)
refined = []
failed = 0

print("🔄 정제 시작...\n")
for i, idx in enumerate(indices):
    s = ds_orig[idx]
    r = refine(s[q_col], s[a_col])
    
    if r and r.get("question") and r.get("answer"):
        refined.append({"instruction": r["question"], "output": r["answer"]})
    else:
        failed += 1
    
    if (i + 1) % 20 == 0:
        print(f"  [{i+1}/200] 성공 {len(refined)} / 실패 {failed}")
    time.sleep(0.3)

print(f"\n✅ 정제 완료: {len(refined)}건")

In [ ]:
# 정제 데이터 저장
import os
os.makedirs("./outputs", exist_ok=True)

with open("./outputs/refined_data.json", "w", encoding="utf-8") as f:
    json.dump(refined, f, ensure_ascii=False, indent=2)

print(f"💾 저장: ./outputs/refined_data.json ({len(refined)}건)")
print(f"\n📋 정제 샘플 2개:")
for i, d in enumerate(refined[:2]):
    print(f"\n[{i+1}] Q: {d['instruction'][:100]}")
    print(f"    A: {d['output'][:150]}...")

---
## 2️⃣ 정제 데이터로 재학습

**주의**: `model`은 Part 4에서 이미 원본 데이터로 학습된 상태.  
새 어댑터로 다시 시작하기 위해 **Part 3 상태로 초기화** 필요.

In [ ]:
# 기존 모델 해제
import gc
del model, trainer
gc.collect()
import torch
torch.cuda.empty_cache()

print("✅ Part 4 model 해제 완료")
print(f"현재 VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")

In [ ]:
# Part 3처럼 EXAONE 4.0 + QLoRA 다시 로드
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training, get_peft_model

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)

print("🔄 EXAONE 4.0 재로드 중...")
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"], quantization_config=bnb_config, device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_config)
print("✅ QLoRA 재적용 완료")
model.print_trainable_parameters()

In [ ]:
# 정제 데이터로 Dataset 변환 + 토크나이징
from datasets import Dataset

SYSTEM = "당신은 대한민국 법률 전문 AI 어시스턴트입니다."

def preprocess_refined(s):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": s["instruction"]},
        {"role": "assistant", "content": s["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    tok = tokenizer(text, truncation=True, max_length=1024, padding=False)
    tok["labels"] = tok["input_ids"].copy()
    return tok

ds_refined = Dataset.from_list(refined)
tokenized_refined = ds_refined.map(preprocess_refined, remove_columns=ds_refined.column_names)
print(f"✅ 토크나이징: {len(tokenized_refined)}건")

In [ ]:
# 학습 실행 (50스텝)
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

args = TrainingArguments(
    output_dir="./exaone4-law-refined-tmp",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    bf16=True,
    optim="paged_adamw_8bit",
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    seed=42,
)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, pad_to_multiple_of=8)

trainer = Trainer(model=model, args=args, train_dataset=tokenized_refined, data_collator=collator)

print("🏋️ 정제 데이터 학습 시작!\n")
result_refined = trainer.train()
print(f"\n✅ 완료! train_loss: {result_refined.training_loss:.4f}")

---
## 3️⃣ 3가지 응답 나란히 비교

In [ ]:
# 정제 학습본으로 5개 질문 응답 생성
model.eval()

refined_answers = []
for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f"[{i}/5] 생성 중...")
    refined_answers.append(generate_fast(model, tokenizer, q))

print("✅ 정제 학습본 응답 5건 완료")

In [ ]:
# 3가지 비교: 베이스라인 vs 원본학습 vs 정제학습
질문 = TEST_QUESTIONS[0]
베이스라인 = baseline_results[0]["baseline_answer"]
원본학습 = finetuned_results[0]["finetuned_raw"]
정제학습 = refined_answers[0]

print(f"[질문] {질문}\n")
for 이름, 응답 in [("1️⃣ 베이스라인 (학습 없음)", 베이스라인),
                  ("2️⃣ 원본 데이터 학습", 원본학습),
                  ("3️⃣ 정제 데이터 학습", 정제학습)]:
    print("=" * 55)
    print(이름)
    print("=" * 55)
    print(응답[:300])
    print()

---
## 4️⃣ LLM-as-Judge 채점

GPT-4o-mini가 4가지 기준 × 10점 = **총 40점**으로 평가

In [ ]:
def judge(question, answer):
    prompt = f"""당신은 대한민국 법률 전문가입니다.

[질문]
{question}

[답변]
{answer[:800]}

4가지 기준 각 1~10점. JSON만 출력:
{{"법률_정확성": N, "핵심_포함": N, "오류_여부": N, "실용성": N, "총점": N}}"""

    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200, temperature=0,
        )
        text = resp.choices[0].message.content.strip()
        if "```" in text:
            text = text.split("```")[1].replace("json", "").strip()
        return json.loads(text)
    except:
        return None

# 5개 질문 × 3개 모델 = 15번 채점
judge_results = []
print("🧑‍⚖️ Judge 채점 중...\n")

for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f"  [{i}/5] {q[:30]}...")
    j_base = judge(q, baseline_results[i-1]["baseline_answer"])
    j_raw = judge(q, finetuned_results[i-1]["finetuned_raw"])
    j_ref = judge(q, refined_answers[i-1])
    judge_results.append({
        "question": q, "base": j_base, "raw": j_raw, "refined": j_ref,
    })
    time.sleep(0.5)

print("\n✅ 채점 완료")

---
## 5️⃣ 최종 비교표

In [ ]:
# 평균 점수 계산
def 평균(key):
    vals = [r[key]["총점"] for r in judge_results if r[key]]
    return sum(vals) / len(vals) if vals else 0

base_avg = 평균("base")
raw_avg = 평균("raw")
refined_avg = 평균("refined")

print("=" * 55)
print("📊 최종 비교 (총점 40 만점)")
print("=" * 55)
print(f"1️⃣ 베이스라인:      {base_avg:.1f} / 40")
print(f"2️⃣ 원본 데이터 학습: {raw_avg:.1f} / 40  ({raw_avg-base_avg:+.1f})")
print(f"3️⃣ 정제 데이터 학습: {refined_avg:.1f} / 40  ({refined_avg-base_avg:+.1f})")
print()

if refined_avg > raw_avg + 1:
    print(f"✅ 정제 학습이 원본 학습보다 {refined_avg-raw_avg:+.1f}점 우수!")
    print("   → 데이터 품질이 모델 성능을 결정한다는 핵심 교훈 확인")
elif raw_avg > refined_avg + 1:
    print(f"🔵 원본 학습이 유리 → 정제 데이터 품질/양 보강 필요")
else:
    print(f"🟡 비슷한 수준 → 더 많은 데이터 또는 에폭 필요")

In [ ]:
# 결과 저장
comparison = {
    "baseline_avg": base_avg,
    "raw_trained_avg": raw_avg,
    "refined_trained_avg": refined_avg,
    "details": [
        {
            "question": r["question"],
            "base_score": r["base"]["총점"] if r["base"] else 0,
            "raw_score": r["raw"]["총점"] if r["raw"] else 0,
            "refined_score": r["refined"]["총점"] if r["refined"] else 0,
        }
        for r in judge_results
    ],
}

with open("./outputs/comparison_results.json", "w", encoding="utf-8") as f:
    json.dump(comparison, f, ensure_ascii=False, indent=2)

# 어댑터 저장
trainer.save_model("./exaone4-law-refined")
tokenizer.save_pretrained("./exaone4-law-refined")

print("💾 저장 완료:")
print("  ./exaone4-law-refined/ (정제 학습 어댑터)")
print("  ./outputs/comparison_results.json (비교 결과)")

---
## 🎓 Part 5 완료

### 3가지 어댑터 비교

| 학습 방식 | 어댑터 경로 | 특징 |
|---|---|---|
| Baseline | (없음) | 학습 전 |
| 원본 학습 | `./exaone4-law-raw/` | 특정 사건 패턴 |
| **정제 학습** | **`./exaone4-law-refined/`** | **법률 원칙 중심** |

### 핵심 교훈

> **"적은 양의 고품질 데이터가 많은 양의 저품질 데이터를 이긴다."**

### 🔜 Part 6 예고

- 두 어댑터 비교 저장
- HuggingFace Hub 업로드 (선택)
- 최종 추론 파이프라인
- 실전 배포 워크플로우